# TrainLM: LLM Training & MFU Benchmark on Colab TPU v5e-1

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_v5e1_MFU_Benchmark.ipynb)

Standard Hugging Face UX — `TrainLMTrainer` handles BF16 placement, whole-step `torch.compile`, `DeferAndFuse` eager fallback, and `chunked_linear` loss automatically underneath:

```python
trainer = TrainLMTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=TrainLMTrainingArguments(
        output_dir="./out",
        per_device_train_batch_size=8,
        sequence_length=1024,
        max_steps=100,
        bf16=True,
        accelerator="native_tpu",
        torch_compile=True,
        loss_implementation="chunked_linear",
    ),
)
trainer.train()
```

The cells below benchmark **SmolLM2-135M**, **SmolLM2-360M**, and **Qwen2.5-0.5B** on **Cloud TPU v5e-1** (16 GB HBM, 197 TFLOP/s BF16 peak), separating **Window 1 (cold compile)** from **Windows 2+ (steady-state MFU)**.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, distribution, version

BRANCH = "main"  # Change to a PR branch if testing pre-merge.
REPO_URL = "https://github.com/Dhiraj309/TrainLM.git"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", f"trainlm @ git+{REPO_URL}@{BRANCH}"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "huggingface-hub>=1.5,<2",
        "PyYAML>=6,<7",
        "transformers>=5.15,<6",
        "datasets>=3,<5",
        "tokenizers>=0.21,<1",
    ],
    check=True,
)

direct_url = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
SOURCE_REVISION = direct_url.get("vcs_info", {}).get("commit_id")
PACKAGES = {}
for pkg in ("trainlm", "torch", "torch-tpu", "libtpu", "transformers", "datasets"):
    try:
        PACKAGES[pkg] = version(pkg)
    except PackageNotFoundError:
        PACKAGES[pkg] = None

print("source_revision=", SOURCE_REVISION)
print("packages=", json.dumps(PACKAGES, indent=2))

In [ ]:
# Select which models and modes to benchmark one by one on TPU v5e-1 (16 GB HBM).
MODELS_TO_RUN = ("smollm2-135m", "smollm2-360m", "qwen2.5-0.5b")
RUN_EAGER = True          # Measure eager baseline on SmolLM2-135M (and fallback if compile fails).
RUN_COMPILED = True       # Measure torch_compile=True on all selected models.
RUN_PROFILER = True       # Capture top 10 CPU/host ops on SmolLM2-135M eager.
STEPS = 25                # 25 steps / 5 logging_steps -> Window 1 (cold) + Windows 2-5 (steady-state).
LOGGING_STEPS = 5
USE_SYNTHETIC_DATA = True # True isolates compute/dispatch MFU; False streams TinyStories text.
TPU_VM_MACHINE_TYPE = None

In [ ]:
import gc
import tempfile
from pathlib import Path
from statistics import mean
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trainlm import TrainLMTrainer, TrainLMTrainingArguments
from trainlm.optimization import probe_torchtpu_splash_kernel
from trainlm.runtime.torchtpu_discovery import discover_tpu, normalize_worker_report
from trainlm.training import TrainerCallback


MODEL_SPECS = {
    "smollm2-135m": {
        "model_id": "HuggingFaceTB/SmolLM2-135M",
        "seq_len": 1024, "batch_size": 8, "grad_accum": 1,
        "grad_ckpt": False, "chunked_loss": False,
    },
    "smollm2-360m": {
        "model_id": "HuggingFaceTB/SmolLM2-360M",
        "seq_len": 512, "batch_size": 8, "grad_accum": 1,
        "grad_ckpt": False, "chunked_loss": True,
    },
    "qwen2.5-0.5b": {
        "model_id": "Qwen/Qwen2.5-0.5B",
        "seq_len": 512, "batch_size": 4, "grad_accum": 1,
        "grad_ckpt": False, "chunked_loss": True,
    },
}


class _FixedShapeDataset(TorchDataset):
    def __init__(self, *, vocab_size, seq_len, length):
        g = torch.Generator().manual_seed(42)
        self._rows = torch.randint(0, vocab_size, (length, seq_len), generator=g, dtype=torch.long)

    def __len__(self):
        return int(self._rows.shape[0])

    def __getitem__(self, idx):
        ids = self._rows[idx]
        return {"input_ids": ids, "labels": ids.clone()}


def _build_dataset(tok, vocab_size, seq_len, total_rows):
    if USE_SYNTHETIC_DATA:
        return _FixedShapeDataset(vocab_size=vocab_size, seq_len=seq_len, length=max(64, total_rows))
    from datasets import load_dataset
    stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    buf, rows = [], []
    for item in stream:
        buf.extend(tok(item["text"], add_special_tokens=False)["input_ids"])
        if tok.eos_token_id is not None:
            buf.append(int(tok.eos_token_id))
        while len(buf) >= seq_len:
            rows.append(torch.tensor(buf[:seq_len], dtype=torch.long))
            buf = buf[seq_len:]
            if len(rows) >= total_rows:
                break
        if len(rows) >= total_rows:
            break
    stacked = torch.stack(rows)
    ds = _FixedShapeDataset(vocab_size=vocab_size, seq_len=seq_len, length=1)
    ds._rows = stacked
    return ds


class _LiveMFUPrinter(TrainerCallback):
    def __init__(self, label, total_steps):
        self.label = label
        self.total_steps = total_steps

    def on_metrics(self, state, control, metrics):
        del control
        if "loss" not in metrics:
            return
        tps = metrics.get("train_tokens_per_second")
        mfu = metrics.get("non_embedding_mfu")
        wait = metrics.get("dataloader_wait_fraction")
        perf = f" | {tps:,.1f} tok/s" if tps is not None else ""
        if mfu is not None:
            perf += f" | MFU {mfu:.2%}"
        if wait is not None:
            perf += f" | wait {wait:.1%}"
        print(f"[{self.label}] step {state.step}/{self.total_steps} | loss={metrics['loss']:.4f}{perf}", flush=True)


def _tpu_mem_gib():
    out = {}
    tpu_mod = getattr(torch, "tpu", None)
    for k, fn_name in (("allocated_gib", "memory_allocated"), ("peak_gib", "max_memory_allocated")):
        fn = getattr(tpu_mod, fn_name, None)
        if callable(fn):
            try:
                out[k] = round(float(fn()) / (1024 ** 3), 3)
            except Exception:
                pass
    return out


def _cleanup_tpu():
    gc.collect()
    tpu_mod = getattr(torch, "tpu", None)
    for fn_name in ("empty_cache", "reset_peak_memory_stats"):
        fn = getattr(tpu_mod, fn_name, None)
        if callable(fn):
            try:
                fn()
            except Exception:
                pass


def _summarize(perf, *, compile_enabled, grad_ckpt):
    windows = perf.get("recent_windows") or []
    cold = windows[0] if windows else {}
    steady = windows[1:] if len(windows) > 1 else windows
    s_tps = mean(w["train_tokens_per_second"] for w in steady) if steady else perf.get("tokens_per_second", 0.0)
    s_mfu = mean(w.get("non_embedding_mfu", 0.0) for w in steady) if steady else perf.get("non_embedding_mfu", 0.0)
    s_lmfu = mean(w.get("logits_inclusive_mfu", 0.0) for w in steady) if steady else perf.get("logits_inclusive_mfu", 0.0)
    s_wait = mean(w.get("dataloader_wait_fraction", 0.0) for w in steady) if steady else perf.get("dataloader_wait_fraction", 0.0)
    causes = []
    if s_wait >= 0.2:
        causes.append(f"high_dataloader_wait ({s_wait:.1%})")
    if not compile_enabled and (s_mfu or 0.0) < 0.20:
        causes.append("eager_python_dispatch_and_unfused_ops")
    if grad_ckpt:
        causes.append("activation_checkpoint_recompute_overhead")
    if (s_mfu or 0.0) < 0.15 and s_wait < 0.2:
        causes.append("small_batch_or_fp32_adamw_memory_bandwidth_bound")
    return {
        "cold_window_1": {
            "tokens_per_second": round(float(cold.get("train_tokens_per_second", 0.0)), 2),
            "non_embedding_mfu_pct": round(float(cold.get("non_embedding_mfu", 0.0)) * 100.0, 2),
            "wall_seconds": round(float(cold.get("train_wall_seconds", 0.0)), 3),
        },
        "steady_state": {
            "windows_count": len(steady),
            "tokens_per_second": round(float(s_tps), 2),
            "non_embedding_mfu_pct": round(float(s_mfu or 0.0) * 100.0, 2),
            "logits_inclusive_mfu_pct": round(float(s_lmfu or 0.0) * 100.0, 2),
            "dataloader_wait_fraction": round(float(s_wait), 4),
        },
        "bottleneck_hint": perf.get("bottleneck_hint"),
        "bottleneck_causes": causes or ["compute_or_kernel_bound"],
        "recent_windows": windows,
    }


def _run_one(key, spec, *, compile_enabled, profile_ops=False):
    label = f"{key}-{'compiled' if compile_enabled else 'eager'}"
    print(f"\n=== Running {label} ({spec['model_id']}) ===", flush=True)
    _cleanup_tpu()
    tok = AutoTokenizer.from_pretrained(spec["model_id"])
    model = AutoModelForCausalLM.from_pretrained(spec["model_id"])
    vocab_size = int(getattr(model.config, "vocab_size", len(tok)))
    ds = _build_dataset(tok, vocab_size, spec["seq_len"], spec["batch_size"] * spec["grad_accum"] * STEPS)

    out_dir = Path(tempfile.mkdtemp(prefix=f"trainlm-{label}-"))
    trainer = TrainLMTrainer(
        model=model,
        processing_class=tok,
        train_dataset=ds,
        callbacks=[_LiveMFUPrinter(label, STEPS)],
        args=TrainLMTrainingArguments(
            output_dir=out_dir,
            max_steps=STEPS,
            sequence_length=spec["seq_len"],
            per_device_train_batch_size=spec["batch_size"],
            gradient_accumulation_steps=spec["grad_accum"],
            learning_rate=1e-4,
            logging_steps=LOGGING_STEPS,
            eval_strategy="no",
            save_strategy="no",
            bf16=True,
            accelerator="native_tpu",
            torch_compile=compile_enabled,
            gradient_checkpointing=spec["grad_ckpt"],
            loss_implementation="chunked_linear" if spec["chunked_loss"] else "auto",
            logits_chunk_size=1024 if spec["chunked_loss"] else None,
        ),
    )
    trainer.train()
    perf = trainer.performance_summary()
    result = _summarize(perf, compile_enabled=compile_enabled, grad_ckpt=spec["grad_ckpt"])
    result.update({
        "model_id": spec["model_id"],
        "parameters": sum(p.numel() for p in model.parameters()),
        "tokens_per_step": spec["seq_len"] * spec["batch_size"] * spec["grad_accum"],
        "compile_enabled": compile_enabled,
        "gradient_checkpointing": spec["grad_ckpt"],
        "chunked_linear_loss": spec["chunked_loss"],
        "hbm": _tpu_mem_gib(),
    })

    if profile_ops:
        try:
            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU], record_shapes=False) as prof:
                batch = {k: v.to(trainer.runtime.device) for k, v in next(iter(trainer.engine.train_dataloader)).items()}
                with trainer.runtime.autocast():
                    out = trainer.model(**batch)
                out.loss.backward()
                trainer.optimizer.step()
                trainer.optimizer.zero_grad(set_to_none=True)
                trainer.runtime.synchronize()
            result["top_cpu_ops"] = prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=12)
        except Exception as exc:
            result["top_cpu_ops_error"] = f"{type(exc).__name__}: {exc}"

    del trainer, model
    _cleanup_tpu()
    return result


BENCHMARK_RESULTS = {}
for idx, key in enumerate(MODELS_TO_RUN):
    spec = MODEL_SPECS[key]
    if RUN_EAGER and (idx == 0 or not RUN_COMPILED):
        try:
            BENCHMARK_RESULTS[f"{key}_eager"] = _run_one(key, spec, compile_enabled=False, profile_ops=(RUN_PROFILER and idx == 0))
        except Exception as exc:
            BENCHMARK_RESULTS[f"{key}_eager"] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
    if RUN_COMPILED:
        try:
            BENCHMARK_RESULTS[f"{key}_compiled"] = _run_one(key, spec, compile_enabled=True)
        except Exception as exc:
            BENCHMARK_RESULTS[f"{key}_compiled"] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            if not RUN_EAGER or f"{key}_eager" not in BENCHMARK_RESULTS:
                try:
                    BENCHMARK_RESULTS[f"{key}_eager_fallback"] = _run_one(key, spec, compile_enabled=False)
                except Exception as exc2:
                    BENCHMARK_RESULTS[f"{key}_eager_fallback"] = {"status": "failed", "error": f"{type(exc2).__name__}: {exc2}"}

TPU_DISCOVERY = normalize_worker_report(discover_tpu(torch, verify_allocation=False, include_properties=True))
print("\n=== v5e-1 MFU Summary ===")
for name, item in BENCHMARK_RESULTS.items():
    if "steady_state" not in item:
        print(f"  {name:24s} | FAILED: {item.get('error')}")
        continue
    s, c = item["steady_state"], item["cold_window_1"]
    print(
        f"  {name:24s} | tok/step={item['tokens_per_step']:5d} | cold={c['non_embedding_mfu_pct']:5.2f}% "
        f"| steady={s['tokens_per_second']:8.1f} tok/s | MFU={s['non_embedding_mfu_pct']:5.2f}% "
        f"(logits={s['logits_inclusive_mfu_pct']:5.2f}%) | wait={s['dataloader_wait_fraction']:.1%} "
        f"| hbm={item['hbm']} | causes={item['bottleneck_causes']}"
    )
if "top_cpu_ops" in BENCHMARK_RESULTS.get("smollm2-135m_eager", {}):
    print("\n--- Top CPU/Host Ops (SmolLM2-135M Eager) ---")
    print(BENCHMARK_RESULTS["smollm2-135m_eager"]["top_cpu_ops"])

In [ ]:
REPORT = {
    "schema_version": 1,
    "scope": "v5e1_llm_mfu_baseline",
    "source_revision": SOURCE_REVISION,
    "vm_machine_type_entered_by_owner": TPU_VM_MACHINE_TYPE,
    "packages": PACKAGES,
    "tpu_discovery": TPU_DISCOVERY,
    "use_synthetic_data": USE_SYNTHETIC_DATA,
    "results": BENCHMARK_RESULTS,
}
REPORT_PATH = Path.cwd() / "trainlm_v5e1_mfu_benchmark.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
print(f"Saved {REPORT_PATH}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(REPORT_PATH.name))
else:
    files.download(str(REPORT_PATH))